# Camera AI — Specific Defect Detection (multi-class / bounding boxes)
## Dataset: Synthetic Industrial Metal Surface Defects

This notebook explores the **"future work"** AI task from Section 8 of `Camera_AI_Detailed_Project_Reference.docx` — instead of the MVP's binary OK/NG classification, this trains a model to identify *which specific defect* is present, with a bounding box around it if the data supports it.

**Honest note before you start:** this dataset's exact annotation format isn't publicly confirmed. It might come with real bounding box annotations (COCO JSON, Pascal VOC XML, or YOLO `.txt` labels), or it might just be plain image folders like the casting dataset. Rather than assume, this notebook **detects which situation you're in automatically** and adapts:

- **If bounding box annotations exist** → trains a real YOLOv8 object detector (via the `ultralytics` library) that draws boxes around defects.
- **If no annotations exist** → falls back to multi-class classification of the specific defect type (e.g. "scratch" vs "crack" vs "pitted"), which still identifies *what* the defect is, just without a box showing *where*.

Run every cell top to bottom, in order — later cells check which mode was detected and only do meaningful work in the relevant path, so it's safe to run straight through regardless of which case applies to this dataset. On Colab: **Runtime → Change runtime type → GPU** first.

## 1. Install and import libraries

`ultralytics` provides YOLOv8 with a very beginner-friendly API - it handles the training loop, loss functions, and even generates its own evaluation plots internally, which is why it was marked as the "typical tool" for object detection in Section 8's AI task table.

In [ ]:
!pip install kagglehub ultralytics torch torchvision scikit-learn matplotlib seaborn pyyaml --quiet

In [ ]:
import os
import json
import time
import shutil
import random
import xml.etree.ElementTree as ET

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Download the dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tatheerabbas/synthetic-industrial-metal-surface-defects")
print("Path to dataset files:", path)

## 3. Reconnaissance — see what's actually inside

Same principle as the previous notebook: look before writing code that assumes a structure. This prints every folder containing images, and every file that could plausibly be an annotation file (`.yaml`, `.json`, `.xml`).

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

def scan_dataset(root):
    print(f"Scanning: {root}\n")
    print("--- Folders containing images ---")
    for dirpath, dirnames, filenames in os.walk(root):
        images = [f for f in filenames if f.lower().endswith(IMAGE_EXTENSIONS)]
        if images:
            rel = os.path.relpath(dirpath, root)
            print(f"  {rel:50s}  ->  {len(images)} images")

    print("\n--- Possible annotation files found ---")
    counts = {"yaml/yml": 0, "json": 0, "xml": 0}
    for dirpath, dirnames, filenames in os.walk(root):
        for fname in filenames:
            lower = fname.lower()
            if lower.endswith((".yaml", ".yml")):
                counts["yaml/yml"] += 1
            elif lower.endswith(".json"):
                counts["json"] += 1
            elif lower.endswith(".xml"):
                counts["xml"] += 1
    for k, v in counts.items():
        print(f"  {k}: {v} file(s)")

scan_dataset(path)

## 4. Auto-detect the annotation format

This checks, in order: a YOLO-style `data.yaml` (already fully prepared for training), then a COCO-style JSON file (has `images`/`annotations`/`categories` keys), then Pascal VOC-style per-image `.xml` files, and finally falls back to "classification only" if none of those are found.

This exact detection logic was tested against four fake fixtures (one for each case) before being included here, to make sure it correctly identifies each format rather than guessing.

In [ ]:
def detect_annotation_format(root):
    yaml_files, json_files, xml_files = [], [], []
    for dirpath, dirnames, filenames in os.walk(root):
        for fname in filenames:
            lower = fname.lower()
            full = os.path.join(dirpath, fname)
            if lower.endswith((".yaml", ".yml")):
                yaml_files.append(full)
            elif lower.endswith(".json"):
                json_files.append(full)
            elif lower.endswith(".xml"):
                xml_files.append(full)

    # Case 1: a YOLO-style data.yaml already exists
    for yf in yaml_files:
        try:
            with open(yf) as f:
                content = yaml.safe_load(f)
            if isinstance(content, dict) and "names" in content and (
                "train" in content or "path" in content
            ):
                return {"mode": "yolo_ready", "path": yf}
        except Exception:
            continue

    # Case 2: COCO-style JSON annotation file
    for jf in json_files:
        try:
            with open(jf) as f:
                content = json.load(f)
            if isinstance(content, dict) and all(
                k in content for k in ("images", "annotations", "categories")
            ):
                return {"mode": "coco_json", "path": jf}
        except Exception:
            continue

    # Case 3: Pascal VOC style - one .xml file per image
    if xml_files:
        return {"mode": "pascal_voc", "path": xml_files}

    # Case 4: nothing that looks like detection annotations
    return {"mode": "classification_only", "path": None}


detection = detect_annotation_format(path)
ANNOTATION_MODE = detection["mode"]
print("Detected annotation mode:", ANNOTATION_MODE)
if detection["path"]:
    preview = detection["path"] if isinstance(detection["path"], str) else detection["path"][:3]
    print("Reference path(s):", preview)

## 5. Annotation conversion functions

These translate COCO or Pascal VOC annotations into YOLO's plain-text label format (one `.txt` file per image, each line: `class_id x_center y_center width height`, all normalized 0-1). `ultralytics` trains directly from this format, so no matter which annotation style the dataset turns out to use, everything converges to the same training step in Section 7.

Both functions were verified by hand before being included here — a fixed, known bounding box was converted and the output numbers were checked against the formula by hand, not just run without inspection.

In [ ]:
def convert_coco_to_yolo(coco_json_path, images_dir, output_dir):
    with open(coco_json_path) as f:
        coco = json.load(f)

    images_by_id = {img["id"]: img for img in coco["images"]}
    categories = sorted(coco["categories"], key=lambda c: c["id"])
    class_names = [c["name"] for c in categories]
    category_id_to_class_idx = {c["id"]: i for i, c in enumerate(categories)}

    labels_dir = os.path.join(output_dir, "labels")
    imgs_out_dir = os.path.join(output_dir, "images")
    os.makedirs(labels_dir, exist_ok=True)
    os.makedirs(imgs_out_dir, exist_ok=True)

    anns_by_image = {}
    for ann in coco["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    for image_id, image_info in images_by_id.items():
        img_w, img_h = image_info["width"], image_info["height"]
        file_stem = os.path.splitext(image_info["file_name"])[0]

        lines = []
        for ann in anns_by_image.get(image_id, []):
            x, y, w, h = ann["bbox"]  # COCO: top-left x, y, width, height
            x_center = (x + w / 2) / img_w
            y_center = (y + h / 2) / img_h
            norm_w = w / img_w
            norm_h = h / img_h
            class_idx = category_id_to_class_idx[ann["category_id"]]
            lines.append(f"{class_idx} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")

        with open(os.path.join(labels_dir, f"{file_stem}.txt"), "w") as f:
            f.write("\n".join(lines))

        src_img = os.path.join(images_dir, image_info["file_name"])
        if os.path.exists(src_img):
            shutil.copy(src_img, os.path.join(imgs_out_dir, image_info["file_name"]))

    return class_names


def convert_voc_to_yolo(xml_files, images_dir, output_dir):
    class_names = set()
    parsed = []
    for xml_path in xml_files:
        tree = ET.parse(xml_path)
        root_el = tree.getroot()
        filename = root_el.find("filename").text
        size = root_el.find("size")
        img_w = int(size.find("width").text)
        img_h = int(size.find("height").text)

        objects = []
        for obj in root_el.findall("object"):
            name = obj.find("name").text
            class_names.add(name)
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            objects.append((name, xmin, ymin, xmax, ymax))

        parsed.append((filename, img_w, img_h, objects))

    class_names = sorted(class_names)
    class_to_idx = {name: i for i, name in enumerate(class_names)}

    labels_dir = os.path.join(output_dir, "labels")
    imgs_out_dir = os.path.join(output_dir, "images")
    os.makedirs(labels_dir, exist_ok=True)
    os.makedirs(imgs_out_dir, exist_ok=True)

    for filename, img_w, img_h, objects in parsed:
        file_stem = os.path.splitext(filename)[0]
        lines = []
        for name, xmin, ymin, xmax, ymax in objects:
            x_center = ((xmin + xmax) / 2) / img_w
            y_center = ((ymin + ymax) / 2) / img_h
            norm_w = (xmax - xmin) / img_w
            norm_h = (ymax - ymin) / img_h
            class_idx = class_to_idx[name]
            lines.append(f"{class_idx} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")

        with open(os.path.join(labels_dir, f"{file_stem}.txt"), "w") as f:
            f.write("\n".join(lines))

        src_img = os.path.join(images_dir, filename)
        if os.path.exists(src_img):
            shutil.copy(src_img, os.path.join(imgs_out_dir, filename))

    return class_names

## 6. Prepare the YOLO dataset folder (only runs if annotations were found)

This cell only does something if `ANNOTATION_MODE` is `"coco_json"` or `"pascal_voc"` — it converts whichever format was found into a YOLO-ready folder plus a `data.yaml` file describing it. If the mode is already `"yolo_ready"`, this is skipped since the data.yaml already exists. If it's `"classification_only"`, this is skipped entirely and Section 8 (the classification fallback) is what actually runs.

In [ ]:
YOLO_DATA_YAML = None

if ANNOTATION_MODE == "yolo_ready":
    YOLO_DATA_YAML = detection["path"]
    print("Using existing data.yaml:", YOLO_DATA_YAML)

elif ANNOTATION_MODE == "coco_json":
    output_dir = "/content/yolo_dataset" if os.path.exists("/content") else "./yolo_dataset"
    # Assumes images live in the same folder as the annotation json, or one level up -
    # adjust `images_dir` below if scan_dataset's output showed a different location.
    images_dir = os.path.dirname(detection["path"])
    class_names = convert_coco_to_yolo(detection["path"], images_dir, output_dir)

    YOLO_DATA_YAML = os.path.join(output_dir, "data.yaml")
    with open(YOLO_DATA_YAML, "w") as f:
        yaml.dump({
            "path": os.path.abspath(output_dir),
            "train": "images",
            "val": "images",  # NOTE: replace with a real held-out split if you have one
            "names": {i: name for i, name in enumerate(class_names)},
        }, f)
    print("Converted COCO annotations. Classes:", class_names)
    print("data.yaml written to:", YOLO_DATA_YAML)

elif ANNOTATION_MODE == "pascal_voc":
    output_dir = "/content/yolo_dataset" if os.path.exists("/content") else "./yolo_dataset"
    images_dir = os.path.dirname(detection["path"][0])
    class_names = convert_voc_to_yolo(detection["path"], images_dir, output_dir)

    YOLO_DATA_YAML = os.path.join(output_dir, "data.yaml")
    with open(YOLO_DATA_YAML, "w") as f:
        yaml.dump({
            "path": os.path.abspath(output_dir),
            "train": "images",
            "val": "images",
            "names": {i: name for i, name in enumerate(class_names)},
        }, f)
    print("Converted Pascal VOC annotations. Classes:", class_names)
    print("data.yaml written to:", YOLO_DATA_YAML)

else:
    print("No bounding box annotations detected - skipping YOLO setup.")
    print("Proceeding to the multi-class classification fallback in Section 8.")

## 7. Train a YOLOv8 object detector (only runs if annotations were found)

`ultralytics` handles the entire training loop internally - loss functions, optimizer, augmentation, and even generates its own evaluation plots (loss curves, precision-recall curves, confusion matrix) automatically in its output folder. `yolov8n` ("nano") is used since it's the smallest/fastest variant, appropriate for a bootcamp timeline and Colab's free-tier compute.

In [ ]:
if YOLO_DATA_YAML is not None:
    from ultralytics import YOLO

    yolo_model = YOLO("yolov8n.pt")  # pretrained on COCO - transfer learning, same idea as Section 5.1
    train_results = yolo_model.train(
        data=YOLO_DATA_YAML,
        epochs=30,
        imgsz=640,
        batch=16,
        seed=SEED,
        plots=True,
    )
    RUN_DIR = train_results.save_dir
    print("Training complete. Results saved to:", RUN_DIR)
else:
    print("Skipped - no annotations were detected.")

### Visual: training curves and confusion matrix (auto-generated by ultralytics)

`ultralytics` saves several evaluation plots directly into its run folder during training - these are loaded and displayed here rather than recreated manually, since they're already exactly what's needed (loss curves, precision/recall, mAP, and a confusion matrix across all defect classes).

In [ ]:
if YOLO_DATA_YAML is not None:
    results_png = os.path.join(RUN_DIR, "results.png")
    confusion_png = os.path.join(RUN_DIR, "confusion_matrix.png")

    for img_path, title in [(results_png, "Training curves (loss, precision, recall, mAP)"),
                              (confusion_png, "Confusion matrix across defect classes")]:
        if os.path.exists(img_path):
            img = plt.imread(img_path)
            plt.figure(figsize=(10, 7))
            plt.imshow(img)
            plt.axis("off")
            plt.title(title)
            plt.show()
        else:
            print(f"{img_path} not found yet - it's generated at the end of training.")
else:
    print("Skipped - no annotations were detected.")

### Visual: sample predictions with bounding boxes

Runs the trained detector on a handful of validation images and displays the predicted bounding boxes directly on top of them - this is the actual "detection result is a bounding box" output requested for this notebook.

In [ ]:
if YOLO_DATA_YAML is not None:
    with open(YOLO_DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)

    val_images_dir = os.path.join(data_cfg["path"], data_cfg["val"])
    sample_images = [
        os.path.join(val_images_dir, f) for f in os.listdir(val_images_dir)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    ][:6]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, img_path in zip(axes.flat, sample_images):
        result = yolo_model.predict(img_path, verbose=False)[0]
        annotated = result.plot()  # returns an image array with boxes drawn on it
        ax.imshow(annotated[:, :, ::-1])  # BGR -> RGB
        ax.axis("off")
        ax.set_title(os.path.basename(img_path), fontsize=9)

    for ax in axes.flat[len(sample_images):]:
        ax.axis("off")

    plt.suptitle("Sample predictions with bounding boxes")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped - no annotations were detected.")

### Save the trained detector (only runs if annotations were found)

Matches the naming convention from `backend/ai_models/` (Section 11) — same idea as the classification notebooks, just a different file type since YOLO models save as `.pt` weights alongside their own architecture config.

In [ ]:
if YOLO_DATA_YAML is not None:
    MODEL_ID = "metal_defect_detector_v1"
    OUTPUT_DIR = "ai_models"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    best_weights = os.path.join(RUN_DIR, "weights", "best.pt")
    dest_path = os.path.join(OUTPUT_DIR, f"{MODEL_ID}.pt")
    shutil.copy(best_weights, dest_path)

    metadata = {
        "model_id": MODEL_ID,
        "architecture": "yolov8n",
        "task": "object_detection",
        "class_names": data_cfg["names"],
        "input_size": 640,
    }
    with open(os.path.join(OUTPUT_DIR, f"{MODEL_ID}.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    print("Detector weights saved to:", dest_path)

    import shutil as _shutil
    from google.colab import files
    _shutil.make_archive("ai_models_export", "zip", OUTPUT_DIR)
    files.download("ai_models_export.zip")
else:
    print("Skipped - no annotations were detected.")

## 8. Fallback: multi-class defect classification (only runs if NO annotations were found)

If Section 4 detected `"classification_only"`, this section trains a model that identifies the *specific defect type* (not just OK/NG) using the same transfer-learning approach as the earlier binary notebooks, generalized to however many classes actually exist in the dataset's folders. There's no bounding box here since there's nothing in the data to learn box coordinates from - this is a genuine, honest limitation of working without annotation data, not a shortcut.

In [ ]:
if ANNOTATION_MODE == "classification_only":
    # Reuse the train/test-or-single-pool detection logic from the previous notebook
    def find_dir(root, name):
        for dirpath, dirnames, _ in os.walk(root):
            if os.path.basename(dirpath).lower() == name.lower():
                return dirpath
        return None

    train_dir = find_dir(path, "train")
    test_dir = find_dir(path, "test")

    IMG_SIZE = 224
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

    train_transforms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    eval_transforms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    if train_dir is not None:
        full_train_aug = datasets.ImageFolder(train_dir, transform=train_transforms)
        full_train_plain = datasets.ImageFolder(train_dir, transform=eval_transforms)
        test_dataset = datasets.ImageFolder(test_dir, transform=eval_transforms)
        class_names = full_train_aug.classes

        VAL_RATIO = 0.15
        num_samples = len(full_train_aug)
        val_size = int(num_samples * VAL_RATIO)
        indices = torch.randperm(num_samples, generator=torch.Generator().manual_seed(SEED)).tolist()
        train_indices, val_indices = indices[val_size:], indices[:val_size]

        train_dataset = Subset(full_train_aug, train_indices)
        val_dataset = Subset(full_train_plain, val_indices)
    else:
        full_aug = datasets.ImageFolder(path, transform=train_transforms)
        full_plain = datasets.ImageFolder(path, transform=eval_transforms)
        class_names = full_aug.classes

        num_samples = len(full_aug)
        indices = torch.randperm(num_samples, generator=torch.Generator().manual_seed(SEED)).tolist()
        train_end = int(num_samples * 0.70)
        val_end = int(num_samples * 0.85)

        train_dataset = Subset(full_aug, indices[:train_end])
        val_dataset = Subset(full_plain, indices[train_end:val_end])
        test_dataset = Subset(full_plain, indices[val_end:])

    print("Classes found:", class_names)
    print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

    BATCH_SIZE = 32
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
else:
    print("Skipped - annotations were detected, object detection was used instead (Sections 6-7).")

### Build and train the multi-class model

Same MobileNetV2 transfer-learning setup as before, but the final layer now outputs one score per actual defect type instead of just 2 (OK/NG).

In [ ]:
if ANNOTATION_MODE == "classification_only":
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False

    num_classes = len(class_names)
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.classifier.parameters(), lr=1e-4)
    NUM_EPOCHS = 12

    def run_epoch(model, loader, criterion, optimizer=None):
        is_training = optimizer is not None
        model.train() if is_training else model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.set_grad_enabled(is_training):
            for images, labels in loader:
                images, labels = images.to(device), labels.to(device)
                if is_training:
                    optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                if is_training:
                    loss.backward()
                    optimizer.step()
                total_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        return total_loss / total, correct / total

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    BEST_MODEL_PATH = "best_model_temp.pt"

    for epoch in range(1, NUM_EPOCHS + 1):
        start = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"Epoch {epoch}/{NUM_EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | {time.time()-start:.1f}s")

    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    print(f"\nBest validation accuracy: {best_val_acc:.4f}")
else:
    print("Skipped - object detection was used instead.")

### Visual: training curves

In [ ]:
if ANNOTATION_MODE == "classification_only":
    epochs_range = range(1, NUM_EPOCHS + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(epochs_range, history["train_loss"], label="Train loss", marker="o")
    axes[0].plot(epochs_range, history["val_loss"], label="Val loss", marker="o")
    axes[0].set_title("Loss over epochs")
    axes[0].legend()
    axes[1].plot(epochs_range, history["train_acc"], label="Train accuracy", marker="o")
    axes[1].plot(epochs_range, history["val_acc"], label="Val accuracy", marker="o")
    axes[1].set_title("Accuracy over epochs")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("Skipped - object detection was used instead.")

### Evaluate on the test set: confusion matrix, per-class report, and multi-class ROC

With more than 2 classes, a single ROC curve doesn't directly apply - instead this computes one ROC curve per class using the "one-vs-rest" approach (treating each class as the positive class against all others combined), which is the standard way to extend ROC/AUC to multi-class problems.

In [ ]:
if ANNOTATION_MODE == "classification_only":
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = probs.argmax(dim=1).cpu()
            all_labels.extend(labels.numpy())
            all_preds.extend(preds.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    test_accuracy = (all_preds == all_labels).mean()
    print(f"Test accuracy: {test_accuracy:.4f}\n")

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(1.2 * len(class_names) + 3, 1.0 * len(class_names) + 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion matrix — specific defect type (test set)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    print(classification_report(all_labels, all_preds, target_names=class_names))

    # One-vs-rest ROC curve per class
    labels_binarized = label_binarize(all_labels, classes=list(range(len(class_names))))
    plt.figure(figsize=(6, 5))
    for i, cls_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(labels_binarized[:, i], all_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cls_name} (AUC={roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("One-vs-rest ROC curves per defect type")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Skipped - object detection was used instead.")

### Visual: sample predictions

In [ ]:
if ANNOTATION_MODE == "classification_only":
    def denormalize(img_tensor):
        mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
        std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
        return img_tensor * std + mean

    images, labels = next(iter(test_loader))
    images_device = images.to(device)
    with torch.no_grad():
        outputs = model(images_device)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1).cpu()
        confidences = probs.max(dim=1).values.cpu()

    n_show = min(8, len(images))
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for i, ax in enumerate(axes.flat):
        if i >= n_show:
            ax.axis("off")
            continue
        true_label = class_names[labels[i]]
        pred_label = class_names[preds[i]]
        conf = confidences[i].item()
        is_correct = labels[i] == preds[i]
        color = "green" if is_correct else "red"
        img = denormalize(images[i]).permute(1, 2, 0).numpy().clip(0, 1)
        ax.imshow(img)
        ax.set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.2f})", color=color, fontsize=9)
        ax.axis("off")
    plt.suptitle("Sample test predictions (green = correct, red = wrong)")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped - object detection was used instead.")

### Save the multi-class model

In [ ]:
if ANNOTATION_MODE == "classification_only":
    MODEL_ID = "metal_defect_type_classifier_v1"
    OUTPUT_DIR = "ai_models"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    model_path = os.path.join(OUTPUT_DIR, f"{MODEL_ID}.pt")
    torch.save(model.state_dict(), model_path)

    metadata = {
        "model_id": MODEL_ID,
        "architecture": "mobilenet_v2",
        "task": "multi_class_classification",
        "input_size": IMG_SIZE,
        "class_names": class_names,
        "normalize_mean": IMAGENET_MEAN,
        "normalize_std": IMAGENET_STD,
        "test_accuracy": float(test_accuracy),
    }
    with open(os.path.join(OUTPUT_DIR, f"{MODEL_ID}.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    print("Model saved to:", model_path)

    import shutil as _shutil
    from google.colab import files
    _shutil.make_archive("ai_models_export", "zip", OUTPUT_DIR)
    files.download("ai_models_export.zip")
else:
    print("Skipped - object detection was used instead.")

## 9. Why this matters for the reference doc's scope decision

Whichever path actually ran above is worth comparing back to Section 8 (AI tasking scope) of the reference doc, which chose binary OK/NG classification for the MVP specifically because of the data and time constraints described there. This notebook is direct evidence for that reasoning either way:

- **If object detection ran:** notice how much more setup this needed — annotation format detection, a full conversion step, and a different training library entirely — compared to the binary classification notebook's much shorter path from dataset to trained model. Same underlying idea (transfer learning), meaningfully more moving parts.
- **If the classification fallback ran:** this is direct, concrete evidence of the exact risk Section 2.2 of the reference doc warned about — real-world datasets don't always arrive with the annotations a project plan hoped for, and a system's scope sometimes gets decided by what the data actually supports, not just what would be ideal.

Either way, swapping this model into the backend follows the same pattern as before: only `ai_inference.py`'s `classify_image()` (or a new `detect_defects()` function, for the object detection case) needs to change — the API, database, and frontend stay exactly the same, since the `model_id` field in the template schema (Section 9) was designed to make exactly this kind of swap painless.